# 02 - Dataset2 Honeybee Veri Keşfi

Bu notebook, Dataset2 Honeybee veri setindeki görüntü ve YOLO etiket dosyalarını inceler.

Amaç ham veri yapısını, görüntü-etiket eşleşmelerini, sınıf dağılımını ve bounding box özelliklerini kontrol etmektir. Bu aşamada ham veri değiştirilmez, patch metadata üretilmez ve model eğitimi yapılmaz.

## 1. Kütüphaneler

Bu bölümde dosya sistemi işlemleri, tablo analizleri, görüntü okuma ve temel görselleştirme için kullanılan kütüphaneler yüklenir.

In [ ]:
from pathlib import Path
from datetime import datetime
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from PIL import Image
except ImportError as exc:
    raise ImportError("Pillow is required. Install it with: pip install pillow") from exc

warnings.filterwarnings("ignore")

## 2. Ayarlar

Bu bölümde incelenecek veri seti, split listesi, sınıf eşlemesi, patch boyutu adayları ve overwrite ayarları tanımlanır.

In [ ]:
DATASET_NAME = "dataset2_honeybee"
DATASET_DIR_NAME = "dataset2_honeybee"
STAGE_NAME = "01_exploration"
NOTEBOOK_NAME = "02_dataset2_honeybee_exploration"

SPLITS = ["train", "valid", "test"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
CLASS_MAP = {0: "bee", 1: "varroa"}
PATCH_SIZE_CANDIDATES = [48, 64, 80]

RANDOM_STATE = 42
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

## 3. Yardımcı Fonksiyonlar ve Dosya Yolları

Bu bölümde proje kök dizini, veri seti yolu ve bu notebooka ait input-output klasörleri tanımlanır.

Yardımcı fonksiyonlar tablo kaydetme, görsel kaydetme, kısa log yazdırma ve klasör yollarını okunabilir biçimde gösterme işlemleri için kullanılır.

In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        try:
            start_path = Path(__file__).resolve()
        except NameError:
            start_path = Path.cwd().resolve()

    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("Project root could not be found.")


PROJECT_ROOT = find_project_root()
APPROACH_NAME = "traditional_ml"
APPROACH_DIR = PROJECT_ROOT / "approaches" / APPROACH_NAME

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / DATASET_DIR_NAME

NOTEBOOK_DIR = APPROACH_DIR / "notebooks" / STAGE_NAME
OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME

TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for directory in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def relative_path(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def log_section(title):
    print(f"\n===== {title} =====")


def log_output(message, level="INFO"):
    print(f"[{level}] {message}")


def log_dataframe(title, df, max_rows=10, note=None):
    log_section(title)

    if note:
        print(note)

    if df is None or df.empty:
        print("[INFO] DataFrame is empty.")
        return

    preview_df = df.head(max_rows) if max_rows is not None else df
    print(preview_df)

    if max_rows is not None and len(df) > max_rows:
        print(f"... total rows: {len(df)}")


def save_dataframe_csv(df, path, overwrite=False, note=""):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[LOAD] Existing CSV kept: {relative_path(path)}")

        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            log_output(f"[INFO] Existing CSV is empty; using current DataFrame: {relative_path(path)}")
            return df

    df.to_csv(path, index=False, encoding="utf-8-sig")
    log_output(f"[SAVE] CSV saved: {relative_path(path)}")

    return df


def save_figure(fig, path, title="", description="", overwrite=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and not overwrite:
        log_output(f"[SKIP] Existing figure kept: {relative_path(path)}")
        plt.close(fig)
        return path

    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    log_output(f"[SAVE] Figure saved: {relative_path(path)}")

    return path

## 4. Veri Seti Envanteri

Bu bölümde her split için görüntü ve etiket dosyaları listelenir.

Görüntü dosyaları ile etiket dosyalarının eşleşme durumu kontrol edilir ve eksik görüntü ya da eksik etiket kayıtları ayrı tablolar olarak kaydedilir.

In [ ]:
log_section("01 Dataset Exploration Started")
log_output(f"Run timestamp: {RUN_TIMESTAMP}")
log_output(f"Project root: {PROJECT_ROOT}")
log_output(f"Dataset root: {DATASET_ROOT}")
log_output(f"Output directory: {OUTPUT_DIR}")
log_output(f"Class map: {CLASS_MAP}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_ROOT}")


def list_files(split, subdir, extensions=None):
    folder = DATASET_ROOT / split / subdir

    if not folder.exists():
        return []

    files = []

    for path in folder.iterdir():
        if path.is_file() and (extensions is None or path.suffix.lower() in extensions):
            files.append(path)

    return sorted(files)


image_records = []
label_records = []

for split in SPLITS:
    image_files = list_files(split, "images", IMAGE_EXTENSIONS)
    label_files = list_files(split, "labels", {".txt"})

    for path in image_files:
        image_records.append({
            "split": split,
            "stem": path.stem,
            "image_path": relative_path(path),
            "image_suffix": path.suffix.lower(),
            "image_size_bytes": path.stat().st_size,
        })

    for path in label_files:
        label_records.append({
            "split": split,
            "stem": path.stem,
            "label_path": relative_path(path),
            "label_size_bytes": path.stat().st_size,
        })


images_df = pd.DataFrame(image_records)
labels_df = pd.DataFrame(label_records)

file_count_summary_df = pd.DataFrame([
    {
        "split": split,
        "image_count": int((images_df["split"] == split).sum()) if not images_df.empty else 0,
        "label_count": int((labels_df["split"] == split).sum()) if not labels_df.empty else 0,
    }
    for split in SPLITS
])

file_count_summary_df["image_minus_label_count"] = (
    file_count_summary_df["image_count"] - file_count_summary_df["label_count"]
)

matching_records = []
missing_label_records = []
missing_image_records = []

for split in SPLITS:
    image_stems = set(images_df.loc[images_df["split"] == split, "stem"]) if not images_df.empty else set()
    label_stems = set(labels_df.loc[labels_df["split"] == split, "stem"]) if not labels_df.empty else set()

    all_stems = sorted(image_stems | label_stems)
    matched = image_stems & label_stems
    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)

    matching_records.append({
        "split": split,
        "total_stem_count": len(all_stems),
        "matched_image_label_count": len(matched),
        "missing_label_count": len(missing_labels),
        "missing_image_count": len(missing_images),
    })

    for stem in missing_labels:
        missing_label_records.append({
            "split": split,
            "stem": stem,
            "issue": "missing_label",
        })

    for stem in missing_images:
        missing_image_records.append({
            "split": split,
            "stem": stem,
            "issue": "missing_image",
        })


image_label_matching_summary_df = pd.DataFrame(matching_records)
missing_labels_df = pd.DataFrame(
    missing_label_records,
    columns=["split", "stem", "issue"],
)

missing_images_df = pd.DataFrame(
    missing_image_records,
    columns=["split", "stem", "issue"],
)

file_count_summary_df = save_dataframe_csv(
    file_count_summary_df,
    TABLES_DIR / "dataset_file_count_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Dataset file count summary.",
)

image_label_matching_summary_df = save_dataframe_csv(
    image_label_matching_summary_df,
    TABLES_DIR / "image_label_matching_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Image-label matching summary.",
)

missing_labels_df = save_dataframe_csv(
    missing_labels_df,
    TABLES_DIR / "missing_labels.csv",
    overwrite=OVERWRITE_TABLES,
    note="Images without label file.",
)

missing_images_df = save_dataframe_csv(
    missing_images_df,
    TABLES_DIR / "missing_images.csv",
    overwrite=OVERWRITE_TABLES,
    note="Labels without image file.",
)

log_dataframe("Dataset File Count Summary", file_count_summary_df)
log_dataframe("Image-Label Matching Summary", image_label_matching_summary_df)

## 5. Görüntü ve Boş Etiket Kontrolleri

Bu bölümde boş etiket dosyaları, okunamayan görüntüler ve temel görüntü boyutları kontrol edilir.

Bu kontroller veri setindeki yapısal sorunları modelleme aşamasına geçmeden önce görünür hale getirir.

In [ ]:
log_section("02 IMAGE VE EMPTY LABEL KONTROLLERİ")
empty_label_files_df = labels_df[labels_df["label_size_bytes"] == 0].copy() if not labels_df.empty else pd.DataFrame()
if not empty_label_files_df.empty:
    empty_label_files_df["issue"] = "empty_label_file"
image_size_records = []
unreadable_records = []
for _, row in images_df.iterrows():
    path = PROJECT_ROOT / row["image_path"]
    try:
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
        image_size_records.append({**row.to_dict(), "image_width": int(width), "image_height": int(height), "image_mode": mode, "is_readable": True})
    except Exception as exc:
        image_size_records.append({**row.to_dict(), "image_width": np.nan, "image_height": np.nan, "image_mode": "", "is_readable": False})
        unreadable_records.append({"split": row["split"], "stem": row["stem"], "image_path": row["image_path"], "issue": str(exc)})
image_sizes_df = pd.DataFrame(image_size_records)
unreadable_images_df = pd.DataFrame(
    unreadable_records,
    columns=["split", "stem", "image_path", "issue"],
)
if image_sizes_df.empty:
    image_size_summary_df = pd.DataFrame()
else:
    image_size_summary_df = image_sizes_df.groupby("split", as_index=False).agg(image_count=("stem", "count"), readable_image_count=("is_readable", "sum"), unique_size_count=("image_width", lambda s: image_sizes_df.loc[s.index, ["image_width", "image_height"]].drop_duplicates().shape[0]), min_width=("image_width", "min"), max_width=("image_width", "max"), min_height=("image_height", "min"), max_height=("image_height", "max"))
save_dataframe_csv(empty_label_files_df, TABLES_DIR / "empty_label_files.csv", note="Empty label files.")
save_dataframe_csv(image_sizes_df, TABLES_DIR / "image_sizes_detail.csv", note="Readable image size detail.")
save_dataframe_csv(image_size_summary_df, TABLES_DIR / "image_size_summary.csv", note="Image size summary.")
save_dataframe_csv(unreadable_images_df, TABLES_DIR / "unreadable_images.csv", note="Unreadable images.")
log_dataframe("Empty Label Files", empty_label_files_df)
log_dataframe("Image Size Summary", image_size_summary_df)
log_dataframe("Unreadable Images", unreadable_images_df)

## 6. YOLO Etiket Ayrıştırma ve Sınıf Dağılımı

Bu bölümde YOLO etiket satırları ayrıştırılır ve geçerli-geçersiz satırlar belirlenir.

Geçerli etiketler üzerinden sınıf dağılımı hesaplanır.

In [ ]:
log_section("03 YOLO PARSE VE CLASS DAĞILIMI")
def parse_yolo_line(raw_line):
    fields = raw_line.strip().split()
    result = {"raw_line": raw_line.strip(), "fields_count": len(fields), "class_id": np.nan, "class_name": "", "x_center": np.nan, "y_center": np.nan, "bbox_width_norm": np.nan, "bbox_height_norm": np.nan, "bbox_area_norm": np.nan, "x1_norm": np.nan, "y1_norm": np.nan, "x2_norm": np.nan, "y2_norm": np.nan, "parse_error": "", "has_five_fields": False, "is_numeric": False, "class_id_is_integer": False, "class_id_known": False, "coords_in_0_1": False, "positive_width": False, "positive_height": False, "bbox_edges_in_0_1": False, "valid_yolo_row": False}
    if len(fields) != 5:
        result["parse_error"] = "not_five_fields"
        return result
    result["has_five_fields"] = True
    try:
        values = [float(v) for v in fields]
        result["is_numeric"] = True
    except ValueError:
        result["parse_error"] = "non_numeric_value"
        return result
    class_float, x, y, w, h = values
    class_id = int(class_float)
    result.update({"class_id": class_id, "class_name": CLASS_MAP.get(class_id, "unknown"), "x_center": x, "y_center": y, "bbox_width_norm": w, "bbox_height_norm": h, "bbox_area_norm": w * h if pd.notna(w) and pd.notna(h) else np.nan})
    result["class_id_is_integer"] = float(class_id) == class_float
    result["class_id_known"] = class_id in CLASS_MAP
    result["coords_in_0_1"] = all(0 <= v <= 1 for v in [x, y, w, h])
    result["positive_width"] = w > 0
    result["positive_height"] = h > 0
    x1, y1 = x - w / 2, y - h / 2
    x2, y2 = x + w / 2, y + h / 2
    result.update({"x1_norm": x1, "y1_norm": y1, "x2_norm": x2, "y2_norm": y2})
    result["bbox_edges_in_0_1"] = (0 <= x1 <= 1) and (0 <= y1 <= 1) and (0 <= x2 <= 1) and (0 <= y2 <= 1)
    checks = [result["has_five_fields"], result["is_numeric"], result["class_id_is_integer"], result["class_id_known"], result["coords_in_0_1"], result["positive_width"], result["positive_height"], result["bbox_edges_in_0_1"]]
    result["valid_yolo_row"] = bool(all(checks))
    if not result["valid_yolo_row"]:
        failed = []
        for key in ["class_id_is_integer", "class_id_known", "coords_in_0_1", "positive_width", "positive_height", "bbox_edges_in_0_1"]:
            if not result[key]:
                failed.append(key)
        result["parse_error"] = ";".join(failed)
    return result
parsed_records = []
for _, row in labels_df.iterrows():
    label_path = PROJECT_ROOT / row["label_path"]
    if not label_path.exists() or label_path.stat().st_size == 0:
        continue
    lines = label_path.read_text(encoding="utf-8", errors="replace").splitlines()
    for line_number, raw_line in enumerate(lines, start=1):
        if not raw_line.strip():
            continue
        parsed = parse_yolo_line(raw_line)
        parsed_records.append({"split": row["split"], "stem": row["stem"], "label_path": row["label_path"], "line_number": line_number, **parsed})
parsed_yolo_labels_df = pd.DataFrame(parsed_records)
if parsed_yolo_labels_df.empty:
    valid_yolo_labels_df = pd.DataFrame()
    invalid_yolo_rows_df = pd.DataFrame()
else:
    valid_yolo_labels_df = parsed_yolo_labels_df[parsed_yolo_labels_df["valid_yolo_row"] == True].copy()
    invalid_yolo_rows_df = parsed_yolo_labels_df[parsed_yolo_labels_df["valid_yolo_row"] == False].copy()
image_join_cols = ["split", "stem", "image_path", "image_width", "image_height", "is_readable"]
if not valid_yolo_labels_df.empty and not image_sizes_df.empty:
    valid_yolo_labels_df = valid_yolo_labels_df.merge(image_sizes_df[image_join_cols], on=["split", "stem"], how="left")
    valid_yolo_labels_df["bbox_width_px"] = valid_yolo_labels_df["bbox_width_norm"] * valid_yolo_labels_df["image_width"]
    valid_yolo_labels_df["bbox_height_px"] = valid_yolo_labels_df["bbox_height_norm"] * valid_yolo_labels_df["image_height"]
    valid_yolo_labels_df["bbox_area_px"] = valid_yolo_labels_df["bbox_width_px"] * valid_yolo_labels_df["bbox_height_px"]
if not invalid_yolo_rows_df.empty and not image_sizes_df.empty:
    invalid_yolo_rows_df = invalid_yolo_rows_df.merge(image_sizes_df[image_join_cols], on=["split", "stem"], how="left")
yolo_parse_summary_df = pd.DataFrame([{"total_non_empty_label_rows": len(parsed_yolo_labels_df), "valid_yolo_row_count": len(valid_yolo_labels_df), "invalid_yolo_row_count": len(invalid_yolo_rows_df), "valid_yolo_row_ratio": len(valid_yolo_labels_df) / len(parsed_yolo_labels_df) if len(parsed_yolo_labels_df) else np.nan}])
class_distribution_by_split_df = valid_yolo_labels_df.groupby(["split", "class_id", "class_name"], as_index=False).size().rename(columns={"size": "object_count"}) if not valid_yolo_labels_df.empty else pd.DataFrame(columns=["split", "class_id", "class_name", "object_count"])
save_dataframe_csv(parsed_yolo_labels_df, TABLES_DIR / "parsed_yolo_labels_detail.csv", note="Parsed YOLO rows.")
save_dataframe_csv(valid_yolo_labels_df, TABLES_DIR / "valid_yolo_labels_detail.csv", note="Valid YOLO rows.")
save_dataframe_csv(invalid_yolo_rows_df, TABLES_DIR / "invalid_yolo_rows.csv", note="Invalid YOLO rows.")
save_dataframe_csv(yolo_parse_summary_df, TABLES_DIR / "yolo_label_parse_summary.csv", note="YOLO parse summary.")
save_dataframe_csv(class_distribution_by_split_df, TABLES_DIR / "class_distribution_by_split.csv", note="Class distribution by split.")
log_dataframe("YOLO Label Parse Summary", yolo_parse_summary_df)
log_dataframe("Invalid YOLO Rows", invalid_yolo_rows_df)
log_dataframe("Class Distribution by Split", class_distribution_by_split_df)

## 7. Nesne Sayımı ve Bounding Box Özeti

Bu bölümde görüntü başına nesne sayısı ve varroa sayısı hesaplanır.

Ayrıca geçerli bounding box kayıtları için normalize ve piksel ölçeğinde özet istatistikler çıkarılır.

In [ ]:
log_section("04 OBJECT COUNT VE BBOX SUMMARY")
VARROA_CLASS_IDS = [class_id for class_id, class_name in CLASS_MAP.items() if "varroa" in class_name.lower()]
log_output(f"Varroa class ids: {VARROA_CLASS_IDS}")
object_count_records = []
for _, img_row in images_df.iterrows():
    split = img_row["split"]
    stem = img_row["stem"]
    subset = valid_yolo_labels_df[(valid_yolo_labels_df["split"] == split) & (valid_yolo_labels_df["stem"] == stem)] if not valid_yolo_labels_df.empty else pd.DataFrame()
    varroa_subset = subset[subset["class_id"].isin(VARROA_CLASS_IDS)] if not subset.empty else pd.DataFrame()
    object_count_records.append({"split": split, "stem": stem, "image_path": img_row["image_path"], "object_count": len(subset), "varroa_count": len(varroa_subset), "has_any_object": len(subset) > 0, "has_varroa": len(varroa_subset) > 0, "has_zero_varroa": len(varroa_subset) == 0, "is_empty_label_file": bool(((empty_label_files_df["split"] == split) & (empty_label_files_df["stem"] == stem)).any()) if not empty_label_files_df.empty else False, "has_invalid_yolo_row": bool(((invalid_yolo_rows_df["split"] == split) & (invalid_yolo_rows_df["stem"] == stem)).any()) if not invalid_yolo_rows_df.empty else False})
object_count_per_image_detail_df = pd.DataFrame(object_count_records)
object_count_per_image_summary_df = object_count_per_image_detail_df.groupby("split", as_index=False).agg(image_count=("stem", "count"), images_with_any_object=("has_any_object", "sum"), images_with_zero_object=("object_count", lambda s: int((s == 0).sum())), images_with_varroa=("has_varroa", "sum"), images_with_zero_varroa=("has_zero_varroa", "sum"), total_objects=("object_count", "sum"), total_varroa_objects=("varroa_count", "sum"), max_varroa_per_image=("varroa_count", "max")) if not object_count_per_image_detail_df.empty else pd.DataFrame()
bbox_metrics = ["bbox_width_norm", "bbox_height_norm", "bbox_area_norm", "bbox_width_px", "bbox_height_px", "bbox_area_px"]
if valid_yolo_labels_df.empty:
    bbox_summary_by_class_df = pd.DataFrame()
else:
    summary_parts = []
    for metric in bbox_metrics:
        part = valid_yolo_labels_df.groupby(["class_id", "class_name"])[metric].agg(["count", "mean", "min", "median", "max"]).reset_index()
        part = part.rename(columns={"count": f"{metric}_count", "mean": f"{metric}_mean", "min": f"{metric}_min", "median": f"{metric}_median", "max": f"{metric}_max"})
        summary_parts.append(part)
    bbox_summary_by_class_df = summary_parts[0]
    for part in summary_parts[1:]:
        bbox_summary_by_class_df = bbox_summary_by_class_df.merge(part, on=["class_id", "class_name"], how="outer")
save_dataframe_csv(object_count_per_image_detail_df, TABLES_DIR / "object_count_per_image_detail.csv", note="Object count per image detail.")
save_dataframe_csv(object_count_per_image_summary_df, TABLES_DIR / "object_count_per_image_summary.csv", note="Object count per image summary.")
save_dataframe_csv(bbox_summary_by_class_df, TABLES_DIR / "bbox_summary_by_class.csv", note="BBox summary by class.")
log_dataframe("Object Count per Image Summary", object_count_per_image_summary_df)
log_dataframe("Bounding Box Summary by Class", bbox_summary_by_class_df)

## 8. Uç Bounding Box Adayları

Bu bölümde veri setine özgü uç bounding box adayları belirlenir.

Adaylar, bounding box genişliği, yüksekliği ve alanı için normalize ve piksel ölçeğindeki dağılımlar kullanılarak hesaplanır.

In [ ]:
log_section("05 DATASET-SPECIFIC EXTREME BBOX THRESHOLDS")
varroa_labels_df = valid_yolo_labels_df[valid_yolo_labels_df["class_id"].isin(VARROA_CLASS_IDS)].copy() if not valid_yolo_labels_df.empty and VARROA_CLASS_IDS else pd.DataFrame()

def compute_extreme_thresholds(df, metrics):
    records = []
    for metric in metrics:
        values = df[metric].dropna() if metric in df.columns else pd.Series(dtype=float)
        if values.empty:
            q005 = q995 = q1 = q3 = iqr = small_iqr_threshold = large_iqr_threshold = small_threshold_used = large_threshold_used = np.nan
        else:
            q005 = values.quantile(0.005)
            q995 = values.quantile(0.995)
            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1
            small_iqr_threshold = q1 - 3.0 * iqr
            large_iqr_threshold = q3 + 3.0 * iqr
            small_threshold_used = max(0.0, min(q005, small_iqr_threshold))
            large_threshold_used = max(q995, large_iqr_threshold)
        records.append({"metric": metric, "q005": q005, "q995": q995, "q1": q1, "q3": q3, "iqr": iqr, "small_iqr_threshold": small_iqr_threshold, "large_iqr_threshold": large_iqr_threshold, "small_threshold_used": small_threshold_used, "large_threshold_used": large_threshold_used, "small_formula": "max(0, min(Q0.005, Q1 - 3*IQR))", "large_formula": "max(Q0.995, Q3 + 3*IQR)"})
    return pd.DataFrame(records)
extreme_bbox_thresholds_df = compute_extreme_thresholds(varroa_labels_df, bbox_metrics)
if varroa_labels_df.empty:
    varroa_labels_with_extreme_flags_df = pd.DataFrame()
    extreme_small_candidates_df = pd.DataFrame()
    extreme_large_candidates_df = pd.DataFrame()
    varroa_bbox_exclusion_candidates_df = pd.DataFrame()
else:
    varroa_labels_with_extreme_flags_df = varroa_labels_df.copy()
    small_flag_columns = []
    large_flag_columns = []
    for _, threshold_row in extreme_bbox_thresholds_df.iterrows():
        metric = threshold_row["metric"]
        small_threshold = threshold_row["small_threshold_used"]
        large_threshold = threshold_row["large_threshold_used"]
        small_col = f"extreme_small_by_{metric}"
        large_col = f"extreme_large_by_{metric}"
        varroa_labels_with_extreme_flags_df[small_col] = False if pd.isna(small_threshold) else (varroa_labels_with_extreme_flags_df[metric] <= small_threshold)
        varroa_labels_with_extreme_flags_df[large_col] = False if pd.isna(large_threshold) else (varroa_labels_with_extreme_flags_df[metric] >= large_threshold)
        small_flag_columns.append(small_col)
        large_flag_columns.append(large_col)
    varroa_labels_with_extreme_flags_df["dataset_specific_extreme_small_candidate"] = varroa_labels_with_extreme_flags_df[small_flag_columns].any(axis=1)
    varroa_labels_with_extreme_flags_df["dataset_specific_extreme_large_candidate"] = varroa_labels_with_extreme_flags_df[large_flag_columns].any(axis=1)
    varroa_labels_with_extreme_flags_df["dataset_specific_varroa_bbox_exclusion_candidate"] = varroa_labels_with_extreme_flags_df["dataset_specific_extreme_small_candidate"] | varroa_labels_with_extreme_flags_df["dataset_specific_extreme_large_candidate"]
    extreme_small_candidates_df = varroa_labels_with_extreme_flags_df[varroa_labels_with_extreme_flags_df["dataset_specific_extreme_small_candidate"]].copy()
    extreme_large_candidates_df = varroa_labels_with_extreme_flags_df[varroa_labels_with_extreme_flags_df["dataset_specific_extreme_large_candidate"]].copy()
    varroa_bbox_exclusion_candidates_df = varroa_labels_with_extreme_flags_df[varroa_labels_with_extreme_flags_df["dataset_specific_varroa_bbox_exclusion_candidate"]].copy()
if DATASET_NAME == "dataset1_varroa":
    image_exclusion_candidates_df = empty_label_files_df.copy()
    if not image_exclusion_candidates_df.empty:
        image_exclusion_candidates_df["exclusion_reason"] = "empty_label_file"
else:
    image_exclusion_candidates_df = pd.DataFrame(columns=["split", "stem", "exclusion_reason"])
invalid_row_exclusion_count = len(invalid_yolo_rows_df)
invalid_row_image_count = invalid_yolo_rows_df[["split", "stem"]].drop_duplicates().shape[0] if not invalid_yolo_rows_df.empty else 0
exclusion_summary_df = pd.DataFrame([{"dataset_name": DATASET_NAME, "valid_varroa_bbox_count": len(varroa_labels_df), "extreme_small_candidate_count": len(extreme_small_candidates_df), "extreme_large_candidate_count": len(extreme_large_candidates_df), "varroa_bbox_exclusion_candidate_count": len(varroa_bbox_exclusion_candidates_df), "empty_label_image_exclusion_count": len(image_exclusion_candidates_df) if DATASET_NAME == "dataset1_varroa" else 0, "invalid_yolo_row_count": invalid_row_exclusion_count, "invalid_row_image_count": invalid_row_image_count, "raw_data_modified": False, "patch_metadata_created": False, "note": "Extreme bbox candidates are computed using dataset-specific symmetric quantile/IQR thresholds. Dataset1 additionally excludes empty-label images. Dataset2 additionally excludes invalid YOLO rows; invalid-row images should not be used as negative sources."}])
save_dataframe_csv(extreme_bbox_thresholds_df, TABLES_DIR / f"{DATASET_NAME}_extreme_bbox_thresholds.csv", note="Dataset-specific extreme bbox thresholds.")
save_dataframe_csv(extreme_small_candidates_df, TABLES_DIR / f"{DATASET_NAME}_extreme_small_review_candidates.csv", note="Dataset-specific extreme-small varroa bbox candidates.")
save_dataframe_csv(extreme_large_candidates_df, TABLES_DIR / f"{DATASET_NAME}_extreme_large_review_candidates.csv", note="Dataset-specific extreme-large varroa bbox candidates.")
save_dataframe_csv(varroa_bbox_exclusion_candidates_df, TABLES_DIR / f"{DATASET_NAME}_varroa_bbox_exclusion_candidates.csv", note="Varroa bbox exclusion candidates.")
save_dataframe_csv(image_exclusion_candidates_df, TABLES_DIR / f"{DATASET_NAME}_image_exclusion_candidates.csv", note="Image-level exclusion candidates.")
save_dataframe_csv(exclusion_summary_df, TABLES_DIR / f"{DATASET_NAME}_exclusion_summary.csv", note="Exclusion summary.")
log_dataframe("Extreme BBox Thresholds", extreme_bbox_thresholds_df, max_rows=None)
log_dataframe("Extreme Small Review Candidates", extreme_small_candidates_df)
log_dataframe("Extreme Large Review Candidates", extreme_large_candidates_df)
log_dataframe("Varroa BBox Exclusion Candidates", varroa_bbox_exclusion_candidates_df)
log_dataframe("Image Exclusion Candidates", image_exclusion_candidates_df)
log_dataframe("Exclusion Summary", exclusion_summary_df)

## 9. Patch Boyutu İncelemesi

Bu bölümde sonraki patch hazırlama aşaması için aday patch boyutları özetlenir.

Hesaplanan değerler yalnızca karar desteği sağlar; bu notebook patch metadata üretmez.

In [ ]:
log_section("06 PATCH SIZE EVIDENCE / EXTERNAL READINESS")
if PATCH_SIZE_CANDIDATES:
    patch_size_records = []
    for patch_size in PATCH_SIZE_CANDIDATES:
        patch_area = patch_size * patch_size
        if not varroa_labels_df.empty:
            median_width = varroa_labels_df["bbox_width_px"].median()
            median_height = varroa_labels_df["bbox_height_px"].median()
            median_area = varroa_labels_df["bbox_area_px"].median()
            patch_size_records.append({"candidate_patch_size": patch_size, "candidate_patch_area_px": patch_area, "valid_varroa_box_count": len(varroa_labels_df), "median_varroa_width_px": median_width, "median_varroa_height_px": median_height, "median_varroa_area_px": median_area, "median_box_area_to_patch_area_ratio": median_area / patch_area if patch_area else np.nan, "median_patch_width_to_box_width_ratio": patch_size / median_width if median_width else np.nan, "median_patch_height_to_box_height_ratio": patch_size / median_height if median_height else np.nan, "note": "Evidence only; patch metadata is not created in exploration."})
        else:
            patch_size_records.append({"candidate_patch_size": patch_size, "candidate_patch_area_px": patch_area, "valid_varroa_box_count": 0, "median_varroa_width_px": np.nan, "median_varroa_height_px": np.nan, "median_varroa_area_px": np.nan, "median_box_area_to_patch_area_ratio": np.nan, "median_patch_width_to_box_width_ratio": np.nan, "median_patch_height_to_box_height_ratio": np.nan, "note": "No valid varroa labels."})
    patch_size_evidence_summary_df = pd.DataFrame(patch_size_records)
    save_dataframe_csv(patch_size_evidence_summary_df, TABLES_DIR / f"{DATASET_NAME}_patch_size_evidence_summary.csv", note="Patch size evidence summary.")
    log_dataframe("Patch Size Evidence Summary", patch_size_evidence_summary_df, max_rows=None)
else:
    external_validation_readiness_summary_df = pd.DataFrame([{"dataset_name": DATASET_NAME, "role": "external_validation_only", "image_count": len(images_df), "label_count": len(labels_df), "valid_yolo_row_count": len(valid_yolo_labels_df), "invalid_yolo_row_count": len(invalid_yolo_rows_df), "empty_label_file_count": len(empty_label_files_df), "unreadable_image_count": len(unreadable_images_df), "patch_preparation_required": False, "model_training_allowed": False, "note": "Dataset3 is reserved for final external validation only."}])
    save_dataframe_csv(external_validation_readiness_summary_df, TABLES_DIR / "external_validation_readiness_summary.csv", note="External validation readiness summary.")
    log_dataframe("External Validation Readiness Summary", external_validation_readiness_summary_df)

## 10. Temel Görseller

Bu bölümde dosya sayıları, sınıf dağılımı, görüntü boyutları ve bounding box dağılımlarını özetleyen görseller oluşturulur.

Görseller bu notebookun `figures` klasörüne kaydedilir.

In [ ]:
log_section("07 CORE FIGURES")
if not file_count_summary_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(file_count_summary_df))
    width = 0.35
    ax.bar(x - width / 2, file_count_summary_df["image_count"], width, label="images")
    ax.bar(x + width / 2, file_count_summary_df["label_count"], width, label="labels")
    ax.set_xticks(x)
    ax.set_xticklabels(file_count_summary_df["split"])
    ax.set_title("File Counts by Split")
    ax.set_xlabel("Split")
    ax.set_ylabel("Count")
    ax.legend()
    save_figure(fig, FIGURES_DIR / "file_counts_by_split.png", "Figure 1 — File Counts by Split", "Her split için image ve label dosya sayıları.")
if not class_distribution_by_split_df.empty:
    pivot = class_distribution_by_split_df.pivot_table(index="split", columns="class_name", values="object_count", aggfunc="sum", fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("Class Distribution by Split")
    ax.set_xlabel("Split")
    ax.set_ylabel("Object Count")
    ax.legend(title="Class")
    save_figure(fig, FIGURES_DIR / "class_distribution_by_split.png", "Figure 2 — Class Distribution by Split", "Split bazında class dağılımı.")
if not image_sizes_df.empty and image_sizes_df["is_readable"].any():
    readable = image_sizes_df[image_sizes_df["is_readable"] == True].copy()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(readable["image_width"], readable["image_height"], alpha=0.5)
    ax.set_title("Image Size Distribution")
    ax.set_xlabel("Width")
    ax.set_ylabel("Height")
    save_figure(fig, FIGURES_DIR / "image_size_distribution.png", "Figure 3 — Image Size Distribution", "Okunabilir görüntülerin width-height dağılımı.")
if not valid_yolo_labels_df.empty and {"bbox_width_px", "bbox_height_px"}.issubset(valid_yolo_labels_df.columns):
    fig, ax = plt.subplots(figsize=(7, 5))
    for class_name, subset in valid_yolo_labels_df.groupby("class_name"):
        ax.scatter(subset["bbox_width_px"], subset["bbox_height_px"], alpha=0.35, label=class_name, s=12)
    ax.set_title("BBox Width-Height Pixels by Class")
    ax.set_xlabel("BBox Width (px)")
    ax.set_ylabel("BBox Height (px)")
    ax.legend()
    save_figure(fig, FIGURES_DIR / "bbox_width_height_pixels_by_class.png", "Figure 4 — BBox Width-Height Pixels by Class", "BBox piksel genişlik-yükseklik dağılımı.")

## 11. Final Durum

Bu bölümde notebookun genel durumu özetlenir.

Eksik dosya, boş etiket, okunamayan görüntü, geçersiz YOLO satırı ve uç bounding box adayları final durum tablosuna eklenir.

In [ ]:
log_section("08 Final Status")

warning_items = {
    "missing_label_count": int(len(missing_labels_df)),
    "missing_image_count": int(len(missing_images_df)),
    "empty_label_file_count": int(len(empty_label_files_df)),
    "unreadable_image_count": int(len(unreadable_images_df)),
    "invalid_yolo_row_count": int(len(invalid_yolo_rows_df)),
    "extreme_small_candidate_count": int(len(extreme_small_candidates_df)) if "extreme_small_candidates_df" in globals() else 0,
    "extreme_large_candidate_count": int(len(extreme_large_candidates_df)) if "extreme_large_candidates_df" in globals() else 0,
    "varroa_bbox_exclusion_candidate_count": int(len(varroa_bbox_exclusion_candidates_df)) if "varroa_bbox_exclusion_candidates_df" in globals() else 0,
    "image_exclusion_candidate_count": int(len(image_exclusion_candidates_df)) if "image_exclusion_candidates_df" in globals() else 0,
}

final_status = "COMPLETED" if sum(warning_items.values()) == 0 else "COMPLETED_WITH_WARNINGS"

final_status_summary_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "stage_name": STAGE_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "final_status": final_status,
    "run_timestamp": RUN_TIMESTAMP,
    **warning_items,
    "raw_data_modified": False,
    "patch_metadata_created": False,
    "feature_extraction_done": False,
    "model_training_done": False,
    "detection_done": False,
}])

final_status_summary_df = save_dataframe_csv(
    final_status_summary_df,
    TABLES_DIR / "final_status_summary.csv",
    overwrite=OVERWRITE_TABLES,
    note="Final status summary.",
)

log_dataframe("Final Status Summary", final_status_summary_df)

log_output(f"Final notebook status: {final_status}")
log_output(f"Tables directory: {TABLES_DIR}")
log_output(f"Figures directory: {FIGURES_DIR}")